[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/patrickallanfaustino/patrickallanfaustino.github.io/blob/main/docs/assets/notebooks/protein_builder.ipynb)

# Protein Builder v17 — preparo de proteína pré-DM

Repara e protona a estrutura e devolve um PDB pronto para o `gmx pdb2gmx`.
Não monta caixa, não adiciona água nem íons, não minimiza — tudo isso acontece
depois, na escada de preparação do GROMACS.

**O notebook é genérico.** Tudo que é específico de uma proteína — estrutura,
cadeias, pH de trabalho, pKa medidos — fica num arquivo JSON por sistema
(`CONFIG_SISTEMA`). Um notebook, um JSON por sistema.

**Etapas**

1. Extração de cadeia(s)
2. Reparo — PDBFixer (resíduos não padrão, átomos e lacunas ausentes, heteroátomos)
3. Refinamento de lacunas — MODELLER, só quando o PDBFixer inseriu resíduos
4. Águas cristalográficas enterradas (opcional) — detecção por área acessível
5. Protonação — PDB2PQR + PROPKA, com sobreposição por pKa medido
6. Águas: separação e conversão para o modelo de água da produção
7. Verificação — tabela de pKa, zona de ambiguidade, pares acoplados, carga
8. Relatório de proveniência

> **O que mudou em relação ao v16**
>
> - **Configuração por sistema.** O JSON guarda o que é conhecimento sobre a
>   *proteína*; o notebook fica igual para todos os sistemas. O JSON vai, sem
>   edição, para o material suplementar.
> - **Sobreposição de protonação.** Quando existe pKa medido, ele tem
>   precedência sobre o PROPKA. Cada sobreposição exige a **fonte**: estado
>   imposto sem referência é só mais um palpite não verificado.
> - **Tabela de pKa.** O v16 registrava *quais* resíduos mudaram de nome, mas
>   não *por quê*. O v17 lê do log do PDB2PQR o pKa de cada resíduo.
> - **Pares acoplados.** Grupos tituláveis próximos — dois carboxilatos, a dupla
>   Asp–His de uma tríade catalítica, um par salino — têm estados
>   interdependentes; um preditor que decide cada resíduo pelo próprio pKa pode
>   errar o par. O v17 destaca os suspeitos.
> - **Carga líquida esperada**, incluindo íons metálicos, para conferir contra o
>   `pdb2gmx`.
> - **Águas cristalográficas enterradas.** `KEEP_WATER` é tudo ou nada; o
>   `gmx solvate` não preenche cavidades internas. O v17 mantém só as águas
>   enterradas, deixa o PDB2PQR orientá-las junto com a rede de ligações de
>   hidrogênio e as reconstrói na geometria exata do modelo de água da produção
>   (3 ou 4 sítios), lida do próprio `.itp`.
> - **Removidos:** análise de caixa; MODELLER AutoModel e carregamento de
>   sequência (reconstruir a cadeia inteira é outra tarefa); protonação pelo
>   OpenMM (segundo caminho, sem tabela de pKa); minimização pelo OpenMM
>   (redundante com a escada de minimização do GROMACS — e alterava as
>   coordenadas antes de elas virarem a referência das restrições posicionais).

## 1. Imports

In [ ]:
import os, json, shutil, platform, subprocess, contextlib
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.PDB import PDBParser, PDBIO, Select

from pdbfixer import PDBFixer
import openmm
from openmm.app import PDBFile, Modeller

pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)

## 2. Configuração

Duas camadas:

- **Esta célula** — valores genéricos do notebook.
- **`CONFIG_SISTEMA`** — JSON com o que é específico da proteína. Os valores do
  arquivo sobrepõem os desta célula.

Chaves aceitas no JSON: `descricao`, `input_structure`, `chain_ids`, `ph`,
`ffout`, `keep_water`, `keep_metals`, `keep_ligands`, `build_missing_loops`,
`drop_terminal_missing`, `loop_refine`, `protonation_overrides`,
`aguas_enterradas`, `limiar_sasa_agua`, `b_max_agua`, `modelo_agua`,
`arquivo_itp_agua`. Chave
desconhecida levanta erro — é a proteção contra erro de digitação que, de outro
modo, seria ignorado em silêncio.

Formato de uma sobreposição:

```json
{"cadeia": "A", "numero": 172, "estado": "GLH",
 "pKa_medido": 6.7, "fonte": "referência do pKa medido"}
```

`cadeia`, `numero`, `estado` e `fonte` são obrigatórios.

In [ ]:
# ---- Sistema ----
# JSON com o que e especifico da proteina. None = so os valores desta celula.
CONFIG_SISTEMA = "1xnb.json"         # mesma pasta do notebook

# No Colab o JSON nao vem junto com o notebook: baixa do repositorio.
if CONFIG_SISTEMA and not Path(CONFIG_SISTEMA).exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/patrickallanfaustino/"
        "patrickallanfaustino.github.io/main/docs/assets/notebooks/" + CONFIG_SISTEMA,
        CONFIG_SISTEMA)

# ---- Entrada ----
INPUT_STRUCTURE = "estrutura.pdb"     # .pdb ou .cif
CHAIN_IDS       = ["A"]

# ---- Heteroatomos ----
KEEP_WATER   = False
KEEP_METALS  = False
KEEP_LIGANDS = False                  # True: cofatores seguem; parametrize no engine

# ---- Aguas cristalograficas enterradas ----
# KEEP_WATER mantem TODAS; AGUAS_ENTERRADAS mantem so as que a proteina envolve.
# As duas juntas nao fazem sentido e levantam erro.
AGUAS_ENTERRADAS    = False
LIMIAR_SASA_AGUA_A2 = 1.0             # area acessivel do O abaixo disto = enterrada (A^2)
B_MAX_AGUA          = None            # descarta aguas com fator B acima disto; None = sem filtro
MODELO_AGUA         = "opc"           # tip3p | spce | opc3 | opc | tip4pew | tip4p2005 | tip4p
ARQUIVO_ITP_AGUA    = None            # .itp da agua do SEU campo de forca (recomendado):
                                      # geometria e nomes de atomo saem dele, nao da tabela

# ---- Reparo ----
BUILD_MISSING_LOOPS   = True          # insere residuos ausentes internos
DROP_TERMINAL_MISSING = True          # nao reconstroi caudas terminais

# ---- Refinamento de lacunas (MODELLER) ----
LOOP_REFINE   = True                  # so age se o PDBFixer inseriu residuos
LOOP_MODELS   = 7
LOOP_MD_LEVEL = "slow"                # "fast" | "slow"

# ---- Protonacao ----
PH    = 7.0
FFOUT = "AMBER"                       # nomes: AMBER (GLH, HID...) | CHARMM (HSD...)
PROTONATION_OVERRIDES = []            # preencha no JSON do sistema, nao aqui

# ---- Verificacao ----
MARGEM_AMBIGUA_PH   = 1.0             # |pKa - pH| abaixo disto: estado incerto
DIST_PAR_ACOPLADO_A = 6.0             # entre atomos titulaveis, em Angstrom

# ---------------------------------------------------------------------------
# Leitura do JSON do sistema
# ---------------------------------------------------------------------------
_CHAVES_SISTEMA = {
    "descricao": None, "input_structure": "INPUT_STRUCTURE", "chain_ids": "CHAIN_IDS",
    "ph": "PH", "ffout": "FFOUT", "keep_water": "KEEP_WATER",
    "keep_metals": "KEEP_METALS", "keep_ligands": "KEEP_LIGANDS",
    "build_missing_loops": "BUILD_MISSING_LOOPS",
    "drop_terminal_missing": "DROP_TERMINAL_MISSING", "loop_refine": "LOOP_REFINE",
    "protonation_overrides": "PROTONATION_OVERRIDES",
    "aguas_enterradas": "AGUAS_ENTERRADAS", "limiar_sasa_agua": "LIMIAR_SASA_AGUA_A2",
    "b_max_agua": "B_MAX_AGUA", "modelo_agua": "MODELO_AGUA",
    "arquivo_itp_agua": "ARQUIVO_ITP_AGUA",
}
DESCRICAO_SISTEMA = ""
if CONFIG_SISTEMA:
    _cfg = json.loads(Path(CONFIG_SISTEMA).read_text())
    _desconhecidas = set(_cfg) - set(_CHAVES_SISTEMA)
    if _desconhecidas:
        raise ValueError(f"Chaves desconhecidas em {CONFIG_SISTEMA}: {sorted(_desconhecidas)}")
    for _chave, _variavel in _CHAVES_SISTEMA.items():
        if _chave in _cfg and _variavel:
            globals()[_variavel] = _cfg[_chave]
    DESCRICAO_SISTEMA = _cfg.get("descricao", "")

# Cada sobreposicao precisa dizer DE ONDE veio o estado.
for _i, _s in enumerate(PROTONATION_OVERRIDES):
    _faltam = {"cadeia", "numero", "estado", "fonte"} - set(_s)
    if _faltam:
        raise ValueError(f"Sobreposicao #{_i} sem {sorted(_faltam)}: {_s}")
    if not str(_s["fonte"]).strip():
        raise ValueError(f"Sobreposicao #{_i} com fonte vazia: {_s}")
OVERRIDES = {(s["cadeia"], int(s["numero"])): s["estado"].upper()
             for s in PROTONATION_OVERRIDES}

if KEEP_WATER and AGUAS_ENTERRADAS:
    raise ValueError("KEEP_WATER e AGUAS_ENTERRADAS sao excludentes: escolha um.")

OUTPUT_DIR = f"output_{Path(INPUT_STRUCTURE).stem}_{''.join(CHAIN_IDS)}"
PROV = {"steps": [], "sistema": DESCRICAO_SISTEMA, "config": CONFIG_SISTEMA}
print(f"Sistema: {DESCRICAO_SISTEMA or INPUT_STRUCTURE}  |  pH {PH}  |  "
      f"{len(OVERRIDES)} sobreposicao(oes)  |  aguas enterradas: "
      f"{'SIM (' + MODELO_AGUA + ')' if AGUAS_ENTERRADAS else 'nao'}")

### Utilitários e diretórios

In [ ]:
_STD_AA = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS",
           "MET","PHE","PRO","SER","THR","TRP","TYR","VAL","HID","HIE","HIP","HSD",
           "HSE","HSP","CYX","CYM","ASH","GLH","LYN","TYM"}
_METALS = {"ZN","MG","MN","FE","FE2","CA","NA","K","CU","CU1","NI","CO","CD","MO","W"}

# Nome no esquema de FF -> residuo de base.
_BASE = {"GLU":"GLU","GLH":"GLU","ASP":"ASP","ASH":"ASP",
         "HIS":"HIS","HID":"HIS","HIE":"HIS","HIP":"HIS","HSD":"HIS","HSE":"HIS","HSP":"HIS",
         "LYS":"LYS","LYN":"LYS","CYS":"CYS","CYM":"CYS","CYX":"CYS",
         "TYR":"TYR","TYM":"TYR","ARG":"ARG"}
# Carga formal da cadeia lateral por nome.
_CARGA = {"ASP":-1,"GLU":-1,"ASH":0,"GLH":0,"LYS":+1,"LYN":0,"ARG":+1,
          "HIP":+1,"HSP":+1,"HID":0,"HIE":0,"HSD":0,"HSE":0,"HIS":0,
          "CYM":-1,"TYM":-1,"CYS":0,"CYX":0,"TYR":0}
# Ions conhecidos. Ferro fica de fora de proposito: o estado de oxidacao
# (heme Fe2+/Fe3+) e decisao da parametrizacao, nao do notebook.
_CARGA_ION = {"NA":+1,"K":+1,"MG":+2,"CA":+2,"ZN":+2,"MN":+2,"CU":+2,"CU1":+1,
              "NI":+2,"CO":+2,"CD":+2,"CL":-1}
# Protons titulaveis. Numa sobreposicao eles sao removidos: o `pdb2gmx -ignh`
# reconstroi os H a partir do NOME do residuo, e um GLU com HE2 no arquivo
# seria incoerente para qualquer outro leitor.
_H_TITULAVEL = {"GLU":{"HE1","HE2"}, "ASP":{"HD1","HD2"}, "HIS":{"HD1","HE2"},
                "LYS":{"HZ1","HZ2","HZ3"}, "CYS":{"HG"}, "TYR":{"HH"}}
# Atomos que carregam o proton titulavel. CYX fica de fora: cisteina em ponte
# dissulfeto nao titula.
_ATOMO_TITULAVEL = {"ASP":("OD1","OD2"), "GLU":("OE1","OE2"), "HIS":("ND1","NE2"),
                    "LYS":("NZ",), "CYS":("SG",), "TYR":("OH",)}
_PROTONADO = {"GLH","ASH","HIP","HSP","LYS","CYS","TYR"}
_ACIDOS = {"ASP", "GLU"}

def is_amino(name): return name.strip().upper() in _STD_AA
def sh(path, *parts): return os.path.join(path, *parts)

def step(msg):
    print("\n" + "=" * 72); print(msg); print("=" * 72)
    PROV["steps"].append(msg)

class ChainSelect(Select):
    def __init__(self, chain_ids): self.chain_ids = set(chain_ids)
    def accept_chain(self, chain): return chain.id in self.chain_ids

def load_structure(path):
    ext = Path(path).suffix.lower()
    if ext in (".cif", ".mmcif"):
        from Bio.PDB import MMCIFParser
        return MMCIFParser(QUIET=True).get_structure("input", path)
    return PDBParser(QUIET=True).get_structure("input", path)

for folder in ["raw", "fixed", "loops", "protonated", "pqr", "logs", "report", "waters"]:
    os.makedirs(sh(OUTPUT_DIR, folder), exist_ok=True)
print("Diretorio:", OUTPUT_DIR)

## 3. Extração de cadeia(s)

In [ ]:
def extract_chains():
    step("[1/5] Extraindo cadeia(s)")
    structure = load_structure(INPUT_STRUCTURE)
    chains = [c.id for c in structure.get_chains()]
    print("Cadeias encontradas:", ", ".join(chains))
    faltando = [c for c in CHAIN_IDS if c not in chains]
    if faltando:
        raise ValueError(f"Cadeia(s) {faltando} ausente(s). Disponiveis: {chains}")
    out = sh(OUTPUT_DIR, "raw", f"chain_{''.join(CHAIN_IDS)}.pdb")
    io = PDBIO(); io.set_structure(structure); io.save(out, ChainSelect(CHAIN_IDS))
    print("OK ->", out); return out

chain = extract_chains()

## 4. Reparo — PDBFixer

Converte resíduos não padrão (MSE→MET), reconstrói átomos e resíduos ausentes
(com opção de descartar caudas terminais) e remove heteroátomos seletivamente.
Registra quais resíduos foram inseridos, para o refinamento da seção 5.

In [ ]:
def _present_residues(topology):
    present = set()
    for chain in topology.chains():
        for res in chain.residues():
            try: present.add((chain.id, int(res.id)))
            except (TypeError, ValueError): pass
    return present

def _contiguous_ranges(residues):
    bychain = defaultdict(list)
    for c, n in residues: bychain[c].append(n)
    ranges = []
    for c, nums in bychain.items():
        nums = sorted(set(nums)); start = prev = nums[0]
        for n in nums[1:]:
            if n == prev + 1: prev = n
            else: ranges.append((c, start, prev)); start = prev = n
        ranges.append((c, start, prev))
    return ranges

def _handle_heterogens(fixer):
    if KEEP_LIGANDS:
        print("KEEP_LIGANDS=True -> cofatores preservados (params no engine).")
        if not KEEP_WATER: fixer.removeHeterogens(keepWater=False)
        return
    m = Modeller(fixer.topology, fixer.positions); to_del = []
    for res in m.topology.residues():
        name = res.name.strip().upper()
        if is_amino(name): continue
        if name in ("HOH","WAT","TIP3","SPC","H2O") and KEEP_WATER: continue
        if name in _METALS and KEEP_METALS: continue
        to_del.append(res)
    if to_del: m.delete(to_del)
    fixer.topology, fixer.positions = m.topology, m.positions

def pdbfixer_stage(pdb_file):
    step("[2/5] PDBFixer (reparo)")
    fixer = PDBFixer(filename=pdb_file)
    fixer.findNonstandardResidues()
    if fixer.nonstandardResidues:
        print("Nao-padrao:", [(r.name, new) for r, new in fixer.nonstandardResidues])
        fixer.replaceNonstandardResidues()
    fixer.findMissingResidues()
    if not BUILD_MISSING_LOOPS:
        fixer.missingResidues = {}
    elif DROP_TERMINAL_MISSING:
        chains = list(fixer.topology.chains())
        for key in list(fixer.missingResidues.keys()):
            ci, ri = key
            if ri == 0 or ri == len(list(chains[ci].residues())):
                del fixer.missingResidues[key]
    present_before = _present_residues(fixer.topology)
    fixer.findMissingAtoms(); fixer.addMissingAtoms()
    inserted = _present_residues(fixer.topology) - present_before
    loop_ranges = _contiguous_ranges(inserted) if inserted else []
    if loop_ranges:
        print("Lacunas internas inseridas:", [f"{c}:{s}-{e}" for c,s,e in loop_ranges])
        PROV["loop_residues"] = [f"{c}:{s}-{e}" for c,s,e in loop_ranges]
    _handle_heterogens(fixer)
    out = sh(OUTPUT_DIR, "fixed", "fixed.pdb")
    with open(out, "w") as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
    print("OK ->", out); return out, loop_ranges

fixed, loop_ranges = pdbfixer_stage(chain)
fixed, loop_ranges

## 5. Refinamento de lacunas (opcional)

Refina **apenas** os resíduos inseridos na seção 4, com o `DOPELoopModel` do
MODELLER. A estrutura resolvida fica fixa. Sem lacunas, o estágio não faz nada —
e nesse caso **não liste o MODELLER nos métodos**: ferramenta citada que não
atuou sugere ao leitor uma reconstrução que não houve. O relatório registra se
houve lacunas (`lacunas_refinadas`).

In [ ]:
def loop_refine_stage(fixed_pdb, loop_ranges):
    if not LOOP_REFINE:
        return fixed_pdb
    if not loop_ranges:
        step("[3/5] Loop refinement (sem lacunas -> nada a refinar)")
        return fixed_pdb
    step("[3/5] Loop refinement (MODELLER DOPELoopModel)")
    print("Refinando:", [f"{c}:{s}-{e}" for c,s,e in loop_ranges])
    from modeller import Environ, Selection
    from modeller.automodel import DOPELoopModel, refine, assess
    work_dir = os.path.dirname(os.path.abspath(fixed_pdb))
    inimodel = Path(fixed_pdb).name
    md_level = refine.slow if LOOP_MD_LEVEL == "slow" else refine.fast

    class _AutoLoop(DOPELoopModel):
        def select_loop_atoms(self):
            segs = [self.residue_range(f"{s}:{c}", f"{e}:{c}") for (c,s,e) in loop_ranges]
            return Selection(*segs)

    env = Environ(); env.io.atom_files_directory = [".", work_dir]
    with open(sh(OUTPUT_DIR, "logs", "loopmodel.log"), "w") as log:
        with contextlib.redirect_stdout(log):
            m = _AutoLoop(env, inimodel=inimodel, sequence="loops",
                          loop_assess_methods=(assess.DOPE,))
            m.loop.starting_model = 1; m.loop.ending_model = LOOP_MODELS
            m.loop.md_level = md_level; m.make()
    outs = [o for o in m.loop.outputs if o.get("failure") is None]
    if not outs:
        print("Aviso: loop refinement falhou; usando PDBFixer."); return fixed_pdb
    outs.sort(key=lambda o: (o.get("DOPE score") is None, o.get("DOPE score")))
    best = outs[0]
    out = sh(OUTPUT_DIR, "loops", "loops_refined.pdb"); shutil.copy(best["name"], out)
    PROV["loop_refinement"] = {"models": LOOP_MODELS, "md_level": LOOP_MD_LEVEL,
                               "best": best["name"], "dope": best.get("DOPE score")}
    print("OK ->", best["name"]); return out

refined = loop_refine_stage(fixed, loop_ranges)
refined

## 5b. Águas cristalográficas enterradas (opcional)

Ativada por `AGUAS_ENTERRADAS`. O `KEEP_WATER` é tudo ou nada, e o `gmx solvate`
preenche o volume livre da caixa, **não cavidades internas** — uma água
estrutural removida aqui não volta.

**Critério.** Uma água é enterrada quando a área acessível ao solvente do seu
oxigênio, calculada **só contra a proteína**, fica abaixo de
`LIMIAR_SASA_AGUA_A2` (Shrake–Rupley, sonda de 1,4 Å). Calcular contra a
proteína isolada é o que importa: águas vizinhas não contam como cobertura. Uma
água que só parece enterrada porque está cercada de outras águas fica de fora — e
o `solvate` vai recolocar água ali.

A tabela lista, para cada água mantida, os átomos polares da proteína a menos de
3,5 Å: é a rede de ligações de hidrogênio que justifica mantê-la.

As águas vão para o PDB2PQR **junto com a proteína**. Isso é deliberado: a
orientação delas e os tautômeros das histidinas vizinhas são otimizados na mesma
rede. Remover a água antes da protonação é o que leva o preditor a escolher o
tautômero para uma rede que não existe.

> **Coordenadas.** O protocolo vem do arquivo de entrada original e a proteína, da
> estrutura reparada. O PDBFixer não move átomos existentes, então os dois
> referenciais coincidem. Se houve refinamento de lacunas, a coluna
> `menor_dist_proteina_A` acusa águas que passaram a colidir com a alça
> reconstruída; essas são descartadas.

In [ ]:
_RAIO_VDW   = {"C": 1.70, "N": 1.55, "O": 1.52, "S": 1.80, "P": 1.80, "SE": 1.90}
_RAIO_O     = 1.52
_SONDA      = 1.40
_NOMES_H2O  = {"HOH", "WAT", "H2O", "DOD", "SOL", "TIP3", "SPC"}
DIST_CHOQUE_A = 2.4          # agua mais perto que isto de um atomo pesado: colisao


def _elemento(linha):
    el = linha[76:78].strip().upper()
    if el:
        return el
    nome = linha[12:16].strip()
    return "".join(ch for ch in nome if ch.isalpha())[:1].upper()


def _pontos_esfera(n=960):
    """Pontos quase uniformes na esfera unitaria (espiral de Fibonacci)."""
    k = np.arange(n) + 0.5
    phi = np.arccos(1.0 - 2.0 * k / n)
    theta = np.pi * (1.0 + 5 ** 0.5) * k
    return np.column_stack([np.cos(theta) * np.sin(phi),
                            np.sin(theta) * np.sin(phi), np.cos(phi)])

_ESFERA = _pontos_esfera()


def _atomos_proteina(pdb, cadeias):
    xyz, raio, rotulo, polar = [], [], [], []
    for l in Path(pdb).read_text().splitlines():
        if not l.startswith(("ATOM", "HETATM")) or l[16] not in (" ", "A"):
            continue
        if l[21] not in cadeias or not is_amino(l[17:20]):
            continue
        el = _elemento(l)
        if el in ("H", "D"):
            continue
        xyz.append([float(l[30:38]), float(l[38:46]), float(l[46:54])])
        raio.append(_RAIO_VDW.get(el, 1.80))
        rotulo.append(f"{l[17:20].strip().capitalize()}{int(l[22:26])}:{l[12:16].strip()}")
        polar.append(el in ("N", "O"))
    return np.array(xyz), np.array(raio), np.array(rotulo), np.array(polar)


def _aguas_cristalograficas(pdb):
    aguas = []
    for l in Path(pdb).read_text().splitlines():
        if not l.startswith(("ATOM", "HETATM")) or l[16] not in (" ", "A"):
            continue
        if l[17:20].strip().upper() not in _NOMES_H2O or _elemento(l) != "O":
            continue
        aguas.append({"id_cristal": f"{l[21]}:{int(l[22:26])}",
                      "xyz": np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])]),
                      "ocupacao": float(l[54:60] or 1.0), "B": float(l[60:66] or 0.0)})
    return aguas


def _sasa_oxigenio(o, prot_xyz, prot_raio):
    """Area acessivel do O de uma agua, so contra a proteina (Shrake-Rupley)."""
    R = _RAIO_O + _SONDA
    perto = np.linalg.norm(prot_xyz - o, axis=1) < R + prot_raio + _SONDA
    area_total = 4.0 * np.pi * R ** 2
    if not perto.any():
        return area_total
    pontos = o + R * _ESFERA
    raio2 = (prot_raio[perto] + _SONDA) ** 2
    d2 = ((pontos[:, None, :] - prot_xyz[perto][None, :, :]) ** 2).sum(-1)
    return float((~(d2 < raio2[None, :]).any(axis=1)).mean() * area_total)


def detectar_aguas_enterradas(pdb_entrada, pdb_proteina):
    step("[4b] Aguas cristalograficas enterradas")
    prot_xyz, prot_raio, rotulos, polar = _atomos_proteina(pdb_proteina, CHAIN_IDS)
    aguas = _aguas_cristalograficas(pdb_entrada)
    print(f"{len(aguas)} agua(s) cristalografica(s) no arquivo de entrada")
    linhas = []
    for a in aguas:
        d = np.linalg.norm(prot_xyz - a["xyz"], axis=1)
        sasa = _sasa_oxigenio(a["xyz"], prot_xyz, prot_raio)
        contatos = [f"{rotulos[k]} ({d[k]:.2f})"
                    for k in np.argsort(d) if polar[k] and d[k] < 3.5]
        motivo = []
        if sasa >= LIMIAR_SASA_AGUA_A2:
            motivo.append("exposta")
        if d.min() < DIST_CHOQUE_A:
            motivo.append("colisao")
        if B_MAX_AGUA is not None and a["B"] > B_MAX_AGUA:
            motivo.append("B alto")
        linhas.append({"id_cristal": a["id_cristal"], "sasa_O_A2": round(sasa, 2),
                       "menor_dist_proteina_A": round(float(d.min()), 2),
                       "B": a["B"], "ocupacao": a["ocupacao"],
                       "n_contatos_polares": len(contatos),
                       "contatos_polares_A": "; ".join(contatos),
                       "mantida": not motivo, "motivo_descarte": ", ".join(motivo),
                       "_xyz": a["xyz"]})
    tabela = pd.DataFrame(linhas)
    return tabela[tabela["mantida"]].reset_index(drop=True), tabela


def gravar_com_aguas(pdb_proteina, enterradas, saida):
    """Proteina + aguas enterradas (so O). O PDB2PQR adiciona e orienta os H.

    As aguas sao renumeradas depois do ultimo residuo da proteina, na primeira
    cadeia: numeracao original de agua costuma colidir com a da proteina.
    """
    linhas = [l for l in Path(pdb_proteina).read_text().splitlines(keepends=True)
              if l.startswith(("ATOM", "HETATM", "TER"))]
    ultimo = max(int(l[22:26]) for l in linhas if l.startswith(("ATOM", "HETATM")))
    cadeia = CHAIN_IDS[0]
    novas, mapa = [], {}
    for k, (_, a) in enumerate(enterradas.iterrows()):
        numero = ultimo + 1 + k
        x, y, z = a["_xyz"]
        novas.append(f"HETATM{90000 + k:5d}  O   HOH {cadeia}{numero:4d}    "
                     f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00{a['B']:6.2f}           O  \n")
        mapa[a["id_cristal"]] = numero
    Path(saida).write_text("".join(linhas) + "".join(novas) + "END\n")
    return mapa


df_aguas = pd.DataFrame()
ENTRADA_PROTONACAO = refined
if AGUAS_ENTERRADAS:
    df_aguas, df_aguas_todas = detectar_aguas_enterradas(INPUT_STRUCTURE, refined)
    df_aguas_todas.drop(columns="_xyz").to_csv(
        sh(OUTPUT_DIR, "waters", "aguas_cristalograficas.csv"), index=False)
    print(f"{len(df_aguas)} enterrada(s) (SASA do O < {LIMIAR_SASA_AGUA_A2} A2)")
    if df_aguas.empty:
        print("\n[!] Nenhuma agua mantida. As 10 mais enterradas e o motivo do descarte:")
        display(df_aguas_todas.sort_values("sasa_O_A2").head(10)[
            ["id_cristal", "sasa_O_A2", "menor_dist_proteina_A", "B",
             "contatos_polares_A", "motivo_descarte"]])
        print("    Se a agua estrutural que voce procura esta na lista como 'exposta' com "
              "SASA pequena,\n    aumente limiar_sasa_agua; se esta como 'colisao', confira "
              "o reparo da estrutura.")
    if not df_aguas.empty:
        ENTRADA_PROTONACAO = sh(OUTPUT_DIR, "waters", "proteina_com_aguas.pdb")
        mapa = gravar_com_aguas(refined, df_aguas, ENTRADA_PROTONACAO)
        df_aguas["numero_novo"] = df_aguas["id_cristal"].map(mapa)
        display(df_aguas.drop(columns=["_xyz", "mantida", "motivo_descarte"]))
    PROV["aguas_enterradas"] = {
        "limiar_sasa_A2": LIMIAR_SASA_AGUA_A2, "b_max": B_MAX_AGUA,
        "mantidas": df_aguas.drop(columns="_xyz").to_dict("records") if not df_aguas.empty else []}
else:
    print("AGUAS_ENTERRADAS = False — secao desativada.")

## 6. Protonação — PDB2PQR + PROPKA, com sobreposição por pKa medido

O PDB2PQR roda o PROPKA, compara cada pKa predito com o pH e grava o estado no
**nome** do resíduo (`--ffout`): GLH, ASH, HID, HIE, HIP, LYN.

Depois, as sobreposições do JSON substituem os estados preditos. Cada uma é
validada: o resíduo tem de existir e ser do mesmo tipo de base — não dá para
transformar um Glu num His por engano, nem aplicar a sobreposição de um sistema
em outro que não tenha aquele resíduo naquela posição.

O pKa que o PDB2PQR **de fato usou** é lido do log que ele grava ao lado do
`.pqr`, não de uma segunda execução do PROPKA, que poderia divergir.

In [ ]:
def _rodar_pdb2pqr(pdb_file):
    saida_pdb = sh(OUTPUT_DIR, "protonated", "protonated_propka.pdb")
    saida_pqr = sh(OUTPUT_DIR, "pqr", "protein.pqr")
    cmd = ["pdb2pqr30", f"--ff={FFOUT}", f"--ffout={FFOUT}", f"--with-ph={PH}",
           "--keep-chain", "--titration-state-method", "propka",
           "--pdb-output", saida_pdb, pdb_file, saida_pqr]
    print("Rodando:", " ".join(cmd))
    with open(sh(OUTPUT_DIR, "logs", "pdb2pqr_stdout.log"), "w") as log:
        subprocess.run(cmd, check=True, stdout=log, stderr=subprocess.STDOUT)
    return saida_pdb, str(Path(saida_pqr).with_suffix(".log"))


def ler_pka(log_path):
    """Le a tabela 'SUMMARY OF THIS PREDICTION' do log do PDB2PQR."""
    linhas, dentro = [], False
    for linha in Path(log_path).read_text().splitlines():
        if "SUMMARY OF THIS PREDICTION" in linha:
            dentro = True; continue
        if not dentro:
            continue
        campos = linha.split()
        if not campos:
            if linhas: break
            continue
        if campos[0] == "Group":
            continue
        try:
            linhas.append({"residuo": campos[0], "numero": int(campos[1]),
                           "cadeia": campos[2], "pKa": float(campos[3]),
                           "pKa_modelo": float(campos[4])})
        except (ValueError, IndexError):
            if linhas: break
    if not linhas:
        raise RuntimeError(f"Tabela de pKa nao encontrada em {log_path}.")
    return pd.DataFrame(linhas)


def aplicar_sobreposicoes(pdb_in, pdb_out, sobreposicoes):
    """Renomeia os residuos sobrepostos e remove seus protons titulaveis."""
    pendentes = dict(sobreposicoes)
    nomes_originais, saida = {}, []
    for linha in Path(pdb_in).read_text().splitlines(keepends=True):
        if linha.startswith(("ATOM", "HETATM")):
            chave = (linha[21], int(linha[22:26]))
            if chave in sobreposicoes:
                atual, novo = linha[17:20].strip(), sobreposicoes[chave]
                if _BASE.get(atual) != _BASE.get(novo):
                    raise ValueError(
                        f"Sobreposicao invalida em {chave}: o residuo e {atual} "
                        f"(base {_BASE.get(atual)}), nao pode virar {novo}.")
                nomes_originais[chave] = atual
                pendentes.pop(chave, None)
                if linha[12:16].strip() in _H_TITULAVEL.get(_BASE[atual], set()):
                    continue                        # proton titulavel: sai
                linha = linha[:17] + f"{novo:>3}" + linha[20:]
        saida.append(linha)
    if pendentes:
        raise ValueError(f"Residuos das sobreposicoes nao encontrados: "
                         f"{sorted(pendentes)}. Confira cadeia e numeracao no JSON do sistema.")
    Path(pdb_out).write_text("".join(saida))
    return nomes_originais


def protonate(pdb_file):
    step("[4/5] Protonacao (PDB2PQR + PROPKA)")
    pdb_propka, log_pka = _rodar_pdb2pqr(pdb_file)
    tabela = ler_pka(log_pka)

    final = sh(OUTPUT_DIR, "protonated", "protonated.pdb")
    if OVERRIDES:
        originais = aplicar_sobreposicoes(pdb_propka, final, OVERRIDES)
        for chave, antes in originais.items():
            depois = OVERRIDES[chave]
            marca = "mantido" if antes == depois else "CORRIGIDO"
            print(f"  {chave[0]}:{chave[1]}  PROPKA -> {antes}   medido -> {depois}"
                  f"   [{marca}]")
    else:
        shutil.copy(pdb_propka, final)
        originais = {}

    PROV["protonation"] = {
        "engine": "pdb2pqr+propka", "ph": PH, "ffout": FFOUT,
        "overrides": PROTONATION_OVERRIDES,
        "estado_propka_nos_sobrepostos": {f"{c}:{n}": v for (c, n), v in originais.items()},
    }
    print("OK ->", final)
    return final, pdb_propka, tabela

protonated, protonated_propka, tabela_pka = protonate(ENTRADA_PROTONACAO)

## 6b. Águas: separação e conversão para o modelo da produção

O PDB2PQR devolve as águas como `WAT` com os dois hidrogênios chamados `HW` —
nomes que o `pdb2gmx` não aceita. Por isso as águas saem do `protonated.pdb`
(que fica só com a proteína, pronto para `pdb2gmx -ignh`) e vão para um
fragmento `.gro` à parte, reconstruídas na geometria do modelo escolhido:

- **O oxigênio** fica na posição cristalográfica.
- **A orientação** — o plano da molécula e a direção da bissetriz — vem do
  PDB2PQR, que a otimizou junto com a rede de ligações de hidrogênio.
- **Comprimento O–H, ângulo H–O–H e sítio virtual** vêm do modelo. Com
  `ARQUIVO_ITP_AGUA`, saem do próprio `.itp` do seu campo de força
  (`[ settles ]` e `[ virtual_sites3 ]`), assim como os nomes de átomo e de
  resíduo. Sem ele, de uma tabela interna.

**Use o `ARQUIVO_ITP_AGUA`.** Portes diferentes do mesmo modelo usam nomes
diferentes para o sítio virtual (`MW`, `EPW`, `M`). Nome divergente vira aviso do
`grompp`, e aviso suprimido com `-maxwarn` esconde o próximo erro de verdade.

In [ ]:
# r_OH (A), angulo H-O-H (graus), d_OM (A; None = 3 sitios)
_GEOMETRIA_AGUA = {
    "tip3p":     (0.9572,  104.52, None),
    "spce":      (1.0000,  109.47, None),
    "opc3":      (0.97888, 109.47, None),
    "opc":       (0.8724,  103.6,  0.1594),
    "tip4pew":   (0.9572,  104.52, 0.1250),
    "tip4p2005": (0.9572,  104.52, 0.1546),
    "tip4p":     (0.9572,  104.52, 0.1500),
}


def geometria_da_agua():
    """(r_OH, angulo, d_OM, nomes_de_atomo, nome_de_residuo) do modelo escolhido."""
    if ARQUIVO_ITP_AGUA:
        secao, nomes, resname, doh, dhh, a_vs = None, [], "SOL", None, None, None
        for bruta in Path(ARQUIVO_ITP_AGUA).read_text().splitlines():
            l = bruta.split(";")[0].strip()
            if not l or l.startswith("#"):
                continue
            if l.startswith("["):
                secao = l.strip("[] ").lower()
                continue
            c = l.split()
            if secao == "atoms" and len(c) >= 5:
                nomes.append(c[4]); resname = c[3]
            elif secao == "settles" and len(c) >= 4:
                doh, dhh = float(c[2]), float(c[3])
            elif secao in ("virtual_sites3", "dummies3") and len(c) >= 7:
                a_vs = float(c[5])
        if doh is None:
            raise ValueError(f"[ settles ] nao encontrado em {ARQUIVO_ITP_AGUA}")
        r = doh * 10.0
        ang = 2.0 * np.degrees(np.arcsin(dhh / (2.0 * doh)))
        d_om = 2.0 * a_vs * r * np.cos(np.radians(ang / 2.0)) if a_vs is not None else None
        if len(nomes) != (4 if d_om else 3):
            raise ValueError(f"{ARQUIVO_ITP_AGUA}: {len(nomes)} atomos, esperado "
                             f"{4 if d_om else 3}.")
        return r, ang, d_om, nomes, resname, "itp"
    if MODELO_AGUA not in _GEOMETRIA_AGUA:
        raise ValueError(f"MODELO_AGUA desconhecido: {MODELO_AGUA}. "
                         f"Opcoes: {sorted(_GEOMETRIA_AGUA)}")
    r, ang, d_om = _GEOMETRIA_AGUA[MODELO_AGUA]
    nomes = ["OW", "HW1", "HW2"] + (["MW"] if d_om else [])
    return r, ang, d_om, nomes, "SOL", "tabela"


def reconstruir_agua(o, h1, h2, r, ang, d_om):
    """Geometria do modelo, preservando O e a orientacao dada pelo PDB2PQR."""
    u1 = (h1 - o) / np.linalg.norm(h1 - o)
    u2 = (h2 - o) / np.linalg.norm(h2 - o)
    b = u1 + u2; b /= np.linalg.norm(b)                 # bissetriz
    p = u1 - u2; p -= b * np.dot(p, b); p /= np.linalg.norm(p)
    t = np.radians(ang / 2.0)
    sitios = [o, o + r * (np.cos(t) * b + np.sin(t) * p),
                 o + r * (np.cos(t) * b - np.sin(t) * p)]
    if d_om:
        sitios.append(o + d_om * b)                     # sitio virtual na bissetriz
    return sitios


def separar_e_converter_aguas(pdb_protonado):
    step("[5b] Aguas: separacao e conversao")
    proteina, aguas = [], defaultdict(list)
    for l in Path(pdb_protonado).read_text().splitlines(keepends=True):
        if l.startswith(("ATOM", "HETATM")) and l[17:20].strip().upper() in _NOMES_H2O:
            xyz = np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
            aguas[int(l[22:26])].append((_elemento(l), xyz))
        else:
            proteina.append(l)
    Path(pdb_protonado).write_text("".join(proteina))   # so proteina -> pdb2gmx

    if not aguas:
        print("Nenhuma agua no PDB protonado.")
        return None
    r, ang, d_om, nomes, resname, origem = geometria_da_agua()
    print(f"Modelo: {MODELO_AGUA} ({origem}) | O-H {r:.4f} A | H-O-H {ang:.2f} graus | "
          f"{'4 sitios, O-M ' + format(d_om, '.4f') + ' A' if d_om else '3 sitios'}")

    linhas_gro, n_atomo = [], 0
    for numero in sorted(aguas):
        o = next(x for el, x in aguas[numero] if el == "O")
        hs = [x for el, x in aguas[numero] if el == "H"]
        if len(hs) != 2:
            raise RuntimeError(f"Agua {numero}: {len(hs)} hidrogenio(s) no PDB2PQR.")
        for nome, xyz in zip(nomes, reconstruir_agua(o, hs[0], hs[1], r, ang, d_om)):
            n_atomo += 1
            x, y, z = xyz / 10.0                        # A -> nm
            linhas_gro.append(f"{numero % 100000:5d}{resname:<5s}{nome:>5s}"
                              f"{n_atomo % 100000:5d}{x:8.3f}{y:8.3f}{z:8.3f}\n")

    saida = sh(OUTPUT_DIR, "waters", f"aguas_enterradas_{MODELO_AGUA}.gro")
    Path(saida).write_text(f"Aguas cristalograficas enterradas - {MODELO_AGUA}\n"
                           f"{n_atomo:5d}\n" + "".join(linhas_gro) +
                           "   0.00000   0.00000   0.00000\n")
    PROV["aguas_convertidas"] = {"modelo": MODELO_AGUA, "origem_geometria": origem,
                                 "arquivo_itp": ARQUIVO_ITP_AGUA, "n_aguas": len(aguas),
                                 "residuos": sorted(aguas), "fragmento": saida}
    print(f"{len(aguas)} agua(s) -> {saida}")
    print("A faixa de residuos para proteger no genion e a que mesclar_aguas() "
          "devolve, apos o pdb2gmx.")
    return saida


AGUAS_GRO = separar_e_converter_aguas(protonated) if AGUAS_ENTERRADAS else None

## 7. Verificação dos estados de protonação

Quatro conferências sobre o arquivo final:

1. **Tabela completa** — pKa predito, estado que o PROPKA escolheria, estado
   final e fonte (PROPKA ou medido).
2. **Zona de ambiguidade** — |pKa − pH| < `MARGEM_AMBIGUA_PH`. Ali, o erro
   típico de um preditor (~1 unidade de pH) basta para inverter o estado.
3. **Pares acoplados** — resíduos tituláveis com átomos tituláveis a menos de
   `DIST_PAR_ACOPLADO_A`. A lista completa vai para o CSV; na tela aparecem só
   os suspeitos:
   - **dois ácidos vizinhos ambos protonados** — um par acoplado costuma
     compartilhar *um* próton. É o erro que o v16 cometeu na xilanase BcX;
   - **pares com algum membro na zona de ambiguidade** — tríades Asp–His, pares
     salinos com His, histidinas vizinhas.
4. **Carga líquida esperada** — resíduos pelo nome e íons metálicos conhecidos.
   Ligantes e ferro não entram; o notebook avisa quando existem.

In [ ]:
def residuos_do_pdb(pdb_file):
    """{(cadeia, numero): nome} lido do PDB, na ordem do arquivo."""
    residuos = {}
    for linha in Path(pdb_file).read_text().splitlines():
        if linha.startswith(("ATOM", "HETATM")):
            residuos.setdefault((linha[21], int(linha[22:26])), linha[17:20].strip())
    return residuos


def tabela_de_estados(tabela, pdb_final):
    """Estado de cada residuo titulavel.

    Quando a sobreposicao informa `pKa_medido`, a margem e a ambiguidade sao
    calculadas com o valor MEDIDO. Julgar pelo pKa do PROPKA um residuo que ja foi
    corrigido pela literatura so repetiria o alarme que motivou a correcao.
    """
    finais = residuos_do_pdb(pdb_final)
    medidos = {(s["cadeia"], int(s["numero"])): s.get("pKa_medido")
               for s in PROTONATION_OVERRIDES}
    linhas = []
    for _, r in tabela.iterrows():
        if r["residuo"] in ("N+", "C-"):
            continue
        chave = (r["cadeia"], int(r["numero"]))
        nome_final = finais.get(chave, "?")
        pka_medido = medidos.get(chave)
        pka_ref = pka_medido if pka_medido is not None else r["pKa"]
        linhas.append({
            "cadeia": r["cadeia"], "numero": int(r["numero"]), "residuo": r["residuo"],
            "pKa_PROPKA": r["pKa"], "pKa_medido": pka_medido, "pKa_modelo": r["pKa_modelo"],
            "erro_PROPKA": (round(r["pKa"] - pka_medido, 2) if pka_medido is not None
                            else None),
            "margem_pH": round(pka_ref - PH, 2),
            "PROPKA_protona": bool(r["pKa"] > PH),
            "estado_final": nome_final,
            "protonado_final": nome_final in _PROTONADO or nome_final == "ARG",
            "fonte": "medido" if chave in OVERRIDES else "PROPKA",
            "ambiguo": abs(pka_ref - PH) < MARGEM_AMBIGUA_PH,
        })
    return pd.DataFrame(linhas)


def pares_acoplados(pdb_final, estados):
    """Pares de residuos titulaveis com atomos titulaveis a menos de DIST_PAR_ACOPLADO_A."""
    atomos, nomes = defaultdict(list), {}
    for linha in Path(pdb_final).read_text().splitlines():
        if not linha.startswith(("ATOM", "HETATM")):
            continue
        nome = linha[17:20].strip()
        base = _BASE.get(nome)
        if nome == "CYX" or base not in _ATOMO_TITULAVEL:
            continue
        if linha[12:16].strip() not in _ATOMO_TITULAVEL[base]:
            continue
        chave = (linha[21], int(linha[22:26]))
        atomos[chave].append([float(linha[30:38]), float(linha[38:46]),
                              float(linha[46:54])])
        nomes[chave] = nome

    ambiguos = {(r["cadeia"], int(r["numero"]))
                for _, r in estados.iterrows() if r["ambiguo"]}
    chaves = sorted(atomos)
    linhas = []
    for i, a in enumerate(chaves):
        xa = np.array(atomos[a])
        for b in chaves[i + 1:]:
            xb = np.array(atomos[b])
            d = float(np.min(np.linalg.norm(xa[:, None, :] - xb[None, :, :], axis=-1)))
            if d >= DIST_PAR_ACOPLADO_A:
                continue
            acidos_protonados = (_BASE[nomes[a]] in _ACIDOS and _BASE[nomes[b]] in _ACIDOS
                                 and nomes[a] in _PROTONADO and nomes[b] in _PROTONADO)
            com_ambiguo = a in ambiguos or b in ambiguos
            linhas.append({"residuo_1": f"{nomes[a]}{a[1]}:{a[0]}",
                           "residuo_2": f"{nomes[b]}{b[1]}:{b[0]}",
                           "distancia_A": round(d, 2),
                           "acidos_ambos_protonados": acidos_protonados,
                           "membro_ambiguo": com_ambiguo,
                           "suspeito": acidos_protonados or com_ambiguo})
    return pd.DataFrame(linhas, columns=["residuo_1", "residuo_2", "distancia_A",
                                         "acidos_ambos_protonados", "membro_ambiguo",
                                         "suspeito"])


def carga_esperada(pdb_final):
    """Carga formal: residuos pelo nome + ions conhecidos.

    Terminais carregados (N+ e C-) se cancelam em cada cadeia, e cadeias com
    ACE/NME nao tem carga terminal; nos dois casos a contribuicao e zero.
    """
    finais = residuos_do_pdb(pdb_final)
    carga, nao_contabilizados = 0, set()
    for nome in finais.values():
        if nome in _CARGA:
            carga += _CARGA[nome]
        elif nome in _CARGA_ION:
            carga += _CARGA_ION[nome]
        elif not is_amino(nome) and nome not in ("HOH", "WAT", "ACE", "NME", "NHE"):
            nao_contabilizados.add(nome)
    return carga, nao_contabilizados


step("[5/5] Verificacao dos estados de protonacao")
df_estados = tabela_de_estados(tabela_pka, protonated)
df_pares = pares_acoplados(protonated, df_estados)
carga, nao_contabilizados = carga_esperada(protonated)

print(f"pH = {PH}  |  {len(df_estados)} residuos titulaveis  |  "
      f"{int(df_estados['ambiguo'].sum())} na zona de ambiguidade")

atencao = df_estados[df_estados["ambiguo"] | (df_estados["fonte"] == "medido")]
if not atencao.empty:
    print("\nResiduos que merecem conferencia contra a literatura:")
    display(atencao)

suspeitos = df_pares[df_pares["suspeito"]] if not df_pares.empty else df_pares
print(f"\n{len(df_pares)} par(es) titulavel(eis) a menos de {DIST_PAR_ACOPLADO_A} A "
      f"| {len(suspeitos)} suspeito(s) — lista completa no CSV")
if not suspeitos.empty:
    display(suspeitos)
    if suspeitos["acidos_ambos_protonados"].any():
        print("[!] Dois acidos vizinhos AMBOS protonados. Estado improvavel: um par "
              "acoplado\n    costuma compartilhar UM proton. Procure pKa medido e "
              "registre no JSON do sistema.")

print(f"\nCarga liquida esperada: {carga:+d} e")
if nao_contabilizados:
    print(f"    Nao contabilizados: {sorted(nao_contabilizados)} — some a carga que a "
          f"parametrizacao\n    atribuir a eles antes de comparar com o pdb2gmx.")
print("Compare com 'Total charge' na saida do pdb2gmx.")

PROV["verificacao"] = {"carga_esperada": carga,
                       "nao_contabilizados_na_carga": sorted(nao_contabilizados),
                       "n_ambiguos": int(df_estados["ambiguo"].sum()),
                       "pares_suspeitos": suspeitos.to_dict("records")}
df_estados.to_csv(sh(OUTPUT_DIR, "report", "estados_protonacao.csv"), index=False)
df_pares.to_csv(sh(OUTPUT_DIR, "report", "pares_acoplados.csv"), index=False)
df_estados

## 8. Relatório de proveniência

In [ ]:
def _versao(modulo, atributo="__version__"):
    try:
        return getattr(__import__(modulo), atributo, "?")
    except Exception:
        return "nao instalado"

def write_report(final_file):
    versions = {"python": platform.python_version(), "openmm": openmm.version.version,
                "pdbfixer": _versao("pdbfixer"), "pdb2pqr": _versao("pdb2pqr"),
                "propka": _versao("propka"), "modeller": _versao("modeller")}
    report = {
        "input": INPUT_STRUCTURE, "chains": CHAIN_IDS, "pH": PH,
        "residue_naming": FFOUT, "loop_refine": LOOP_REFINE,
        "lacunas_refinadas": PROV.get("loop_residues", []),
        "keep": {"water": KEEP_WATER, "metals": KEEP_METALS, "ligands": KEEP_LIGANDS},
        "final_file": final_file,
        "handoff": "gmx pdb2gmx -f <final> -ignh  (estado pelo NOME do residuo)",
        "versions": versions, "provenance": PROV,
    }
    path = sh(OUTPUT_DIR, "report", "preparation_report.json")
    with open(path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    print("Relatorio ->", path)
    return report

report = write_report(protonated)
if CONFIG_SISTEMA:          # o JSON do sistema acompanha os resultados
    shutil.copy(CONFIG_SISTEMA, sh(OUTPUT_DIR, "report", Path(CONFIG_SISTEMA).name))
report["versions"]

## Handoff para o GROMACS

```bash
gmx pdb2gmx -v -f protonated.pdb -o protein.gro -ignh
```

**O `-ignh` descarta todos os hidrogênios do arquivo.** O estado de protonação
chega à topologia pelo **nome do resíduo** (GLH, ASH, HID, HIE, HIP, LYN), que o
`pdb2gmx` usa para escolher a entrada do `.rtp` e reconstruir os hidrogênios.
(A versão anterior deste notebook dizia "preservando os H" — era o contrário.)

Duas conferências depois do `pdb2gmx`, sempre:

```bash
# 1. a carga bate com a esperada da secao 7?
grep "qtot" topol.top | tail -1

# 2. todos os estados nao padrao chegaram na topologia?
#    compare com a coluna estado_final de report/estados_protonacao.csv
awk '/\[ atoms \]/{f=1;next} /\[ /{f=0} f && $4 ~ /^(GLH|ASH|HIP|HID|HIE|LYN|CYM|CYX|TYM)$/ {print $3, $4}' topol.top | sort -un
```

### Com águas enterradas

As águas entram **depois** do `pdb2gmx` e **antes** do `editconf` — no mesmo
referencial cristalográfico da proteína, para serem centralizadas junto com ela:

```bash
gmx pdb2gmx -v -f protonated.pdb -o protein.gro -ignh -water opc
# celula abaixo: mesclar_aguas("protein.gro", "topol.top", AGUAS_GRO, "protein_aguas.gro")
gmx editconf -f protein_aguas.gro -o box.gro -c -d 1.2 -bt dodecahedron
gmx solvate -cp box.gro -cs tip4p.gro -o solvated.gro -p topol.top
# celula abaixo: consolidar_moleculas("topol.top")
gmx grompp -f ions.mdp -c solvated.gro -p topol.top -o ions.tpr
# cole aqui o gmx select que mesclar_aguas() imprime, com a faixa real de residuos
gmx genion -s ions.tpr -n agua_bulk.ndx -o neutralized.gro -p topol.top \
           -pname NA -nname CL -neutral -conc 0.15
```

Três passos que não são opcionais:

- **`consolidar_moleculas`** — depois do `solvate` ficam duas linhas `SOL`
  seguidas no `[ molecules ]` (as enterradas e as do bulk). O `genion` desconta os
  íons de uma linha só, e se escolher a das águas enterradas a contagem fica
  negativa. Juntar as duas numa linha elimina a ambiguidade.
- **`gmx select`** — sem ele, o `genion` sorteia águas para trocar por íons entre
  **todas**, inclusive as enterradas. Um Na⁺ no lugar da água estrutural, no
  interior da proteína, não gera erro nenhum. O comando pronto, com a faixa
  real, é impresso por `mesclar_aguas`.
- **`-water`** do `pdb2gmx` e o `.itp` usado na seção 6b têm de ser o mesmo modelo.

> **Cofatores/substratos** (`keep_ligands`): o PDB2PQR os preserva sem adicionar
> hidrogênios. Parametrize no engine (GAFF2/OpenFF; para heme, use parâmetros
> publicados) e some a carga deles à esperada antes de comparar.

In [ ]:
# ---- Apos o pdb2gmx: mesclar as aguas enterradas ----------------------------
def mesclar_aguas(gro_proteina, top, gro_aguas, gro_saida):
    """Acrescenta as aguas ao .gro do pdb2gmx e a linha SOL ao topol.top."""
    if gro_aguas is None:
        raise ValueError(
            "Nao ha fragmento de aguas (AGUAS_GRO = None). Ou AGUAS_ENTERRADAS estava "
            "desligado\n(confira a chave 'aguas_enterradas' no JSON do sistema), ou a "
            "secao 5b nao manteve\nnenhuma agua (veja a tabela que ela imprime). Sem "
            "aguas, siga direto para o editconf\ncom o .gro do pdb2gmx.")
    if not Path(gro_aguas).exists():
        raise FileNotFoundError(f"Fragmento de aguas nao encontrado: {gro_aguas}")
    P = Path(gro_proteina).read_text().splitlines()
    W = Path(gro_aguas).read_text().splitlines()
    atomos_p, atomos_w = P[2:2 + int(P[1])], W[2:2 + int(W[1])]
    ultimo_res = int(atomos_p[-1][0:5])
    primeiro_w = int(atomos_w[0][0:5])
    novos, n = [], len(atomos_p)
    for l in atomos_w:
        n += 1
        res = ultimo_res + 1 + (int(l[0:5]) - primeiro_w)
        novos.append(f"{res % 100000:5d}{l[5:15]}{n % 100000:5d}{l[20:]}")
    caixa = P[2 + int(P[1])]
    Path(gro_saida).write_text("\n".join([P[0], f"{n:5d}"] + atomos_p + novos + [caixa]) + "\n")

    n_aguas = len({l[0:5] for l in atomos_w})
    texto = Path(top).read_text().rstrip("\n") + f"\nSOL              {n_aguas}\n"
    Path(top).write_text(texto)

    inclui = [l for l in texto.splitlines() if l.strip().startswith("#include") and
              any(m in l.lower() for m in ("tip", "spc", "opc", "water"))]
    print(f"{n_aguas} agua(s) mescladas -> {gro_saida} | residuos "
          f"{ultimo_res + 1} a {ultimo_res + n_aguas}")
    print("Modelo de agua incluido no topol.top:", inclui or "NENHUM — confira o -water")
    a, b = ultimo_res + 1, ultimo_res + n_aguas
    print("\nDepois do solvate e do grompp de ions.tpr, proteja as aguas enterradas com:")
    print(f"  gmx select -s ions.tpr -on agua_bulk.ndx "
          f"-select 'resname SOL and not resnr {a} to {b}'")
    print("  gmx genion -s ions.tpr -n agua_bulk.ndx -o neutralized.gro -p topol.top "
          "-pname NA -nname CL -neutral -conc 0.15")
    return a, b


def consolidar_moleculas(top):
    """Junta linhas consecutivas do mesmo tipo em [ molecules ] (rodar apos o solvate)."""
    linhas = Path(top).read_text().splitlines()
    i = max(k for k, l in enumerate(linhas) if l.strip().lower().startswith("[ molecules"))
    cabeca, corpo = linhas[:i + 1], linhas[i + 1:]
    saida = []
    for l in corpo:
        c = l.split(";")[0].split()
        if len(c) == 2 and saida and saida[-1][0] == c[0]:
            saida[-1][1] += int(c[1])
        elif len(c) == 2:
            saida.append([c[0], int(c[1])])
        else:
            saida.append([l, None])
    texto = cabeca + [f"{a:<16s} {b}" if b is not None else a for a, b in saida]
    Path(top).write_text("\n".join(texto) + "\n")
    print("\n".join(texto[i:]))


# Exemplo (descomente depois de rodar o pdb2gmx no diretorio de trabalho):
A, B = mesclar_aguas("xilanase.gro", "topol.top", AGUAS_GRO, "xilanase_water.gro")
# ... editconf e solvate ...
# consolidar_moleculas("topol.top")